# Exercícios de Ponto Flutuante, armazenamento e erros

## Bloco 01

### Q1 - Épsilon de máquina
Investigue, em Python, o épsilon de máquina para números
de ponto flutuante. 
Determine o menor valor $\varepsilon$ para o qual:
$1+\varepsilon > 1$.

Compare esse valor com:
$\varepsilon_{\mathrm{mach}}.$


Resposta:

Para o tipo `float` do Python, baseado no padrão IEEE 754 de precisão dupla, temos:

$$
\varepsilon_{\mathrm{mach}}
=2^{-52}
\approx
2{,}220446049250313\times10^{-16}.
$$

O menor incremento representável acima de $1$ é:

$$
\varepsilon=
2{,}220446049250313\times10^{-16}.
$$

Ao testar:

$$
1+\frac{\varepsilon_{\mathrm{mach}}}{2}
$$

obtemos:

$$
1+\frac{\varepsilon_{\mathrm{mach}}}{2}=1.
$$

Enquanto:

$$
1+\varepsilon_{\mathrm{mach}}
= 1{,}00000000000000022>1.
$$

 **Interpretação**

Os números de ponto flutuante são representados por um conjunto **discreto de números representáveis**. Portanto, nem todo número real pode ser representado exatamente.

Quando calculamos:

$$
1+\frac{\varepsilon_{\mathrm{mach}}}{2},
$$

o resultado está entre dois números representáveis e, devido ao **arredondamento**, é representado como:

$$
\boxed{1}.
$$

Por outro lado:

$$
1+\varepsilon_{\mathrm{mach}}
$$

corresponde ao próximo número representável acima de $1$:

$$
\boxed{
1+\varepsilon_{\mathrm{mach}}
= 1{,}00000000000000022
}.
$$

Assim, o resultado demonstra o efeito da **precisão finita** e do **arredondamento numérico** na representação de números em ponto flutuante.

 **Conclusão**

O épsilon de máquina representa, nas proximidades de $1$, a escala mínima de separação entre números de ponto flutuante representáveis. Perturbações menores que essa escala podem ser absorvidas pelo arredondamento e não alterar o valor armazenado.

In [6]:
import math
import sys

eps_mach = sys.float_info.epsilon            # Épsilon de máquina para float (IEEE 754, precisão dupla)
print(f"epsilon_mach = {eps_mach:.17e}")
eps = math.nextafter(1.0, math.inf) - 1.0     # Menor incremento representável acima de 1
print(f"epsilon       = {eps:.17e}")

# Testes
a = 1 + eps_mach / 2
b = 1 + eps_mach

print(f"1 + eps_mach/2 = {a:.17e}")
print(f"1 + eps_mach   = {b:.17e}")

print(a == 1.0)
print(b > 1.0)

epsilon_mach = 2.22044604925031308e-16
epsilon       = 2.22044604925031308e-16
1 + eps_mach/2 = 1.00000000000000000e+00
1 + eps_mach   = 1.00000000000000022e+00
True
True


### Truncamento e arredondamento com algarismos significativos

As funções abaixo permitem representar um número $x$ utilizando exatamente
$n$ **algarismos significativos**.

A ideia principal é determinar a **ordem de grandeza** do número e, a partir
dela, calcular um fator de escala que permita deslocar a casa decimal.

---

 1. Determinação da ordem de grandeza

A ordem de grandeza de um número positivo $x$ pode ser obtida por:

$$
k = \left\lfloor \log_{10}|x| \right\rfloor.
$$

Por exemplo, para:

$$
x = 9890{,}9,
$$

temos:

$$
\log_{10}(9890{,}9) \approx 3{,}995,
$$

portanto:

$$
k = \left\lfloor 3{,}995 \right\rfloor = 3.
$$

Isso significa que $x$ está na ordem de $10^3$.

---

 2. Fator de escala

Para representar $x$ com $n$ algarismos significativos, utilizamos:

$$
f = 10^{n-1-k},
$$

em que:

- $n$ = número de algarismos significativos desejados;
- $k$ = ordem de grandeza de $x$;
- $f$ = fator utilizado para deslocar a casa decimal.

Como:

$$
k = \left\lfloor \log_{10}|x| \right\rfloor,
$$

podemos escrever:

$$
f =
10^{n-1-\left\lfloor\log_{10}|x|\right\rfloor}.
$$

Para $x=9890{,}9$ e $n=3$:

$$
f = 10^{3-1-3}=10^{-1}=0{,}1.
$$

Assim:

$$
x f = 9890{,}9 \times 0{,}1 = 989{,}09.
$$

Agora os três algarismos significativos estão na parte inteira.

---

 3. Função de truncamento

```python
def truncar(x, n):
    fator = 10 ** (n - 1 - int(math.floor(math.log10(abs(x)))))
    return math.trunc(x * fator) / fator

In [7]:
import math

p = 9890.9
q = 9887.1
valor_exato = p - q
n = 3  # Número de algarismos significativos

def truncar(x, n):
    fator = 10 ** (n - 1 - int(math.floor(math.log10(abs(x)))))
    return math.trunc(x * fator) / fator

def arredondar(x, n):
    fator = 10 ** (n - 1 - int(math.floor(math.log10(abs(x)))))
    return round(x * fator) / fator

# Truncamento
p_T = truncar(p, n)
q_T = truncar(q, n)
resultado_T = p_T - q_T

# Arredondamento
p_R = arredondar(p, n)
q_R = arredondar(q, n)
resultado_R = p_R - q_R

# Erros absolutos
erro_T = abs(valor_exato - resultado_T)
erro_R = abs(valor_exato - resultado_R)

print("Valor exato:", valor_exato)

print("\nTruncamento:")
print("p =", p_T)
print("q =", q_T)
print("p - q =", resultado_T)
print("Erro absoluto =", erro_T)

print("\nArredondamento:")
print("p =", p_R)
print("q =", q_R)
print("p - q =", resultado_R)
print("Erro absoluto =", erro_R)

Valor exato: 3.7999999999992724

Truncamento:
p = 9890.0
q = 9880.0
p - q = 10.0
Erro absoluto = 6.200000000000728

Arredondamento:
p = 9890.0
q = 9890.0
p - q = 0.0
Erro absoluto = 3.7999999999992724


## Cancelamento numérico

Considere a equação quadrática

$$
x^2-100{,}0001x+0{,}01=0.
$$

Para uma equação do segundo grau

$$
ax^2+bx+c=0,
$$

as raízes são dadas pela fórmula convencional:

$$
x_{1,2}=
\frac{-b\pm\sqrt{\Delta}}{2a},
$$

em que

$$
\Delta=b^2-4ac.
$$

Neste problema:

$$
a=1,\qquad b=-100{,}0001,\qquad c=0{,}01.
$$

O discriminante é

$$
\Delta=(-100{,}0001)^2-4(1)(0{,}01)
=9999{,}98000001.
$$

A fórmula convencional produz:

$$
x_1=
\frac{100{,}0001+99{,}9999}{2}
=100
$$

e

$$
x_2=
\frac{100{,}0001-99{,}9999}{2}
=0{,}0001.
$$

Na segunda raiz ocorre a subtração de dois números muito próximos:

$$
100{,}0001-99{,}9999.
$$

Esse fenômeno é denominado **cancelamento numérico** e pode provocar perda de algarismos significativos.

Uma formulação numericamente mais estável consiste em calcular

$$
q=-\frac{1}{2}
\left(
b+\operatorname{sign}(b)\sqrt{\Delta}
\right),
$$

e então determinar as raízes por

$$
x_1=\frac{q}{a},
\qquad
x_2=\frac{c}{q}.
$$

Para este problema, a solução exata é

$$
x_1=100,
\qquad
x_2=0{,}0001.
$$

O erro absoluto será calculado por

$$
E_a=|x_{\text{exato}}-x_{\text{aproximado}}|.
$$

O objetivo é comparar a fórmula convencional com a formulação numericamente estável.

In [10]:
import math

a,b,c =  1.0, -100.0001, 0.01         # Coeficientes 
delta = b**2 - 4*a*c                  # Discriminante
raiz_delta = math.sqrt(delta)         # Raiz de delta 

# Fórmula convencional
x1_conv = (-b + raiz_delta) / (2*a)
x2_conv = (-b - raiz_delta) / (2*a)

# Formulação numericamente estável
q = -0.5 * (b + math.copysign(raiz_delta, b))
x1_est = q / a
x2_est = c / q

# Valores exatos
x1_exato = 100.0
x2_exato = 0.0001

# Erros absolutos
erro_x1_conv = abs(x1_exato - x1_conv)
erro_x2_conv = abs(x2_exato - x2_conv)

erro_x1_est = abs(x1_exato - x1_est)
erro_x2_est = abs(x2_exato - x2_est)

# Resultados
print("Discriminante:")
print("Delta =", delta)
print("sqrt(Delta) =", raiz_delta)

print("\nFórmula convencional:")
print("x1 =", x1_conv)
print("x2 =", x2_conv)
print("Erro absoluto em x1 =", erro_x1_conv)
print("Erro absoluto em x2 =", erro_x2_conv)

print("\nFormulação numericamente estável:")
print("x1 =", x1_est)
print("x2 =", x2_est)
print("Erro absoluto em x1 =", erro_x1_est)
print("Erro absoluto em x2 =", erro_x2_est)

Discriminante:
Delta = 9999.98000001
sqrt(Delta) = 99.9999

Fórmula convencional:
x1 = 100.0
x2 = 0.00010000000000331966
Erro absoluto em x1 = 0.0
Erro absoluto em x2 = 3.3196508692975857e-15

Formulação numericamente estável:
x1 = 100.0
x2 = 0.0001
Erro absoluto em x1 = 0.0
Erro absoluto em x2 = 0.0


## Erro de truncamento

Considere a série de Taylor da função seno:

$$\sin(x)
= x-\frac{x^3}{3!}
+\frac{x^5}{5!}
-\frac{x^7}{7!}
+\cdots
$$

Para

$$
x=\frac{\pi}{6},
$$

o valor exato é

$$
\sin\left(\frac{\pi}{6}\right)=0{,}5.
$$

A aproximação será realizada utilizando 1, 2 e 3 termos da série.

- 1 termo

Considerando apenas o primeiro termo:

$$
\sin(x)\approx x.
$$

Assim,

$$
\hat{x}_1=
\frac{\pi}{6}
\approx0{,}5235987756.
$$

O erro absoluto é

$$
E_{a,1}
= \left|
0{,}5-\hat{x}_1
\right|.
$$

- 2 termos

Considerando os dois primeiros termos:

$$
\sin(x)
\approx
x-\frac{x^3}{3!}.
$$

Portanto,

$$
\hat{x}_2
= \frac{\pi}{6}
- \frac{(\pi/6)^3}{3!}.
$$

O erro absoluto é

$$
E_{a,2}
= \left|
0{,}5-\hat{x}_2
\right|.
$$

- 3 termos

Considerando três termos:

$$
\sin(x)
\approx
x-\frac{x^3}{3!}
+\frac{x^5}{5!}.
$$

Portanto,

$$
\hat{x}_3
= \frac{\pi}{6}
- \frac{(\pi/6)^3}{3!}
+
\frac{(\pi/6)^5}{5!}.
$$

O erro absoluto é

$$
E_{a,3}
=\left|
0{,}5-\hat{x}_3
\right|.
$$

O objetivo é verificar como o erro de truncamento diminui à medida que novos termos da série de Taylor são adicionados.

In [11]:
import math
 
x = math.pi / 6          # Valor de x
valor_exato = 0.5        # Valor exato de referência  

# Aproximação com 1 termo: sin(x) ≈ x
aprox_1 = x
erro_1 = abs(valor_exato - aprox_1)

# Aproximação com 2 termos: sin(x) ≈ x - x^3/3!
aprox_2 = x - x**3 / math.factorial(3)
erro_2 = abs(valor_exato - aprox_2)

# Aproximação com 3 termos: sin(x) ≈ x - x^3/3! + x^5/5!
aprox_3 = (
    x
    - x**3 / math.factorial(3)
    + x**5 / math.factorial(5)
)
erro_3 = abs(valor_exato - aprox_3)

# Resultados
print("x =", x)
print("Valor exato =", valor_exato)

print("\nAproximação com 1 termo:")
print("Valor =", aprox_1)
print("Erro absoluto =", erro_1)

print("\nAproximação com 2 termos:")
print("Valor =", aprox_2)
print("Erro absoluto =", erro_2)

print("\nAproximação com 3 termos:")
print("Valor =", aprox_3)
print("Erro absoluto =", erro_3)

x = 0.5235987755982988
Valor exato = 0.5

Aproximação com 1 termo:
Valor = 0.5235987755982988
Erro absoluto = 0.023598775598298816

Aproximação com 2 termos:
Valor = 0.49967417939436376
Erro absoluto = 0.00032582060563624005

Aproximação com 3 termos:
Valor = 0.5000021325887924
Erro absoluto = 2.1325887924472653e-06


## Representação de números

Considere os números:

$$81,\qquad 66{,}25,\qquad -0{,}625,\qquad 0{,}533203125. $$

Para cada número, serão obtidas quatro representações:

1. **Binário convencional**;
2. **Binário em ponto flutuante normalizado**;
3. **IEEE--754 precisão simples**, com 32 bits;
4. **IEEE--754 precisão dupla**, com 64 bits.

-  Binário convencional

A conversão para a base 2 é feita separando a parte inteira e a parte fracionária.

Por exemplo:

$$ 81_{10}=1010001_2.$$

Para o número fracionário:

$$
66{,}25_{10}=1000010{,}01_2.
$$

Também temos:

$$
-0{,}625_{10}=-0{,}101_2
$$

e

$$
0{,}533203125_{10}=0{,}100010001_2.
$$

 - Binário em ponto flutuante

A representação normalizada possui a forma

$$
\pm 1,b_1b_2b_3\cdots \times 2^e.
$$

Assim:

$$
81_{10}
= 1{,}010001_2\times2^6
$$

$$
66{,}25_{10}
= 1{,}00001001_2\times2^6
$$

$$ -0{,}625_{10}
=-1{,}01_2\times2^{-1}
$$

e

$$
0{,}533203125_{10}
= 1{,}00010001_2\times2^{-1}.
$$

- IEEE--754 precisão simples

Na precisão simples, são utilizados 32 bits:

$$
\boxed{
1\;|\;8\;|\;23
}
$$

correspondentes a:

$$
\text{Sinal} \;|\; \text{Expoente} \;|\; \text{Fração}.
$$

O expoente utiliza viés:

$$
\text{bias}=127.
$$

A representação normalizada é:

$$
(-1)^S
\left(1+F\right)
2^{E-127}.
$$

- IEEE--754 precisão dupla

Na precisão dupla, são utilizados 64 bits:

$$
\boxed{
1\;|\;11\;|\;52
}
$$

com viés:

$$
\text{bias}=1023.
$$

A representação é:

$$
(-1)^S
\left(1+F\right)
2^{E-1023}.
$$

O objetivo é comparar as diferentes formas de representação do mesmo número.

In [16]:
import struct

def inteiro_para_binario(n):
    if n == 0:
        return "0"

    sinal = ""
    if n < 0:
        sinal = "-"
        n = abs(n)

    return sinal + bin(n)[2:]

def fracao_para_binario(fracao, bits=32):
    resultado = ""

    for _ in range(bits):           # Use '_' para repetir uma operação n vezes, sem usar o valor da sequência, ou 'i'.
        fracao *= 2

        if fracao >= 1:
            resultado += "1"
            fracao -= 1
        else:
            resultado += "0"

        if fracao == 0:
            break

    return resultado

def binario_convencional(x):
    sinal = "-" if x < 0 else ""
    x = abs(x)

    parte_inteira = int(x)
    parte_fracionaria = x - parte_inteira
    bin_inteira = inteiro_para_binario(parte_inteira)

    if parte_fracionaria == 0:
        return sinal + bin_inteira

    bin_fracionaria = fracao_para_binario(parte_fracionaria)

    return sinal + bin_inteira + "," + bin_fracionaria


def ponto_flutuante_binario(x):
    if x == 0:
        return "0"

    sinal = "-" if x < 0 else ""
    x = abs(x)

    binario = binario_convencional(x)

    if "," in binario:
        inteira, fracionaria = binario.split(",")
    else:
        inteira = binario
        fracionaria = ""

    # Número maior ou igual a 1
    if inteira != "0":
        expoente = len(inteira) - 1
        mantissa = inteira[0] + fracionaria

    # Número menor que 1
    else:
        primeiro_1 = fracionaria.find("1")
        expoente = -(primeiro_1 + 1)
        mantissa = "1" + fracionaria[primeiro_1 + 1:]

    return f"{sinal}{mantissa[0]},{mantissa[1:]} x 2^{expoente}"


def ieee754_32(x):
    bits = format(
        # O pack transforma o número x em uma sequência de bytes, com >I ou >F
        # para os digitos mais significativos aparecerem primeiro.
        struct.unpack(">I", struct.pack(">f", x))[0], 
        "032b"
    )

    sinal = bits[0]
    expoente = bits[1:9]
    fracao = bits[9:]

    return bits, sinal, expoente, fracao


def ieee754_64(x):
    bits = format(
        struct.unpack(">Q", struct.pack(">d", x))[0],
        "064b"
    )

    sinal = bits[0]
    expoente = bits[1:12]
    fracao = bits[12:]

    return bits, sinal, expoente, fracao

numeros = [
    81.0,
    66.25,
    -0.625,
    0.533203125
]

for x in numeros:

    print("=" * 70)
    print("Número decimal:", x)

    print("\n(a) Binário convencional:")
    print(binario_convencional(x))

    print("\n(b) Binário em ponto flutuante:")
    print(ponto_flutuante_binario(x))

    bits32, sinal32, expoente32, fracao32 = ieee754_32(x)

    print("\n(c) IEEE-754 precisão simples - 32 bits:")
    print("Cadeia:", bits32)
    print("Sinal:   ", sinal32)
    print("Expoente:", expoente32)
    print("Fração:  ", fracao32)

    bits64, sinal64, expoente64, fracao64 = ieee754_64(x)

    print("\n(d) IEEE-754 precisão dupla - 64 bits:")
    print("Cadeia:", bits64)
    print("Sinal:   ", sinal64)
    print("Expoente:", expoente64)
    print("Fração:  ", fracao64)

    print()

Número decimal: 81.0

(a) Binário convencional:
1010001

(b) Binário em ponto flutuante:
1, x 2^6

(c) IEEE-754 precisão simples - 32 bits:
Cadeia: 01000010101000100000000000000000
Sinal:    0
Expoente: 10000101
Fração:   01000100000000000000000

(d) IEEE-754 precisão dupla - 64 bits:
Cadeia: 0100000001010100010000000000000000000000000000000000000000000000
Sinal:    0
Expoente: 10000000101
Fração:   0100010000000000000000000000000000000000000000000000

Número decimal: 66.25

(a) Binário convencional:
1000010,01

(b) Binário em ponto flutuante:
1,01 x 2^6

(c) IEEE-754 precisão simples - 32 bits:
Cadeia: 01000010100001001000000000000000
Sinal:    0
Expoente: 10000101
Fração:   00001001000000000000000

(d) IEEE-754 precisão dupla - 64 bits:
Cadeia: 0100000001010000100100000000000000000000000000000000000000000000
Sinal:    0
Expoente: 10000000101
Fração:   0000100100000000000000000000000000000000000000000000

Número decimal: -0.625

(a) Binário convencional:
-0,101

(b) Binário em ponto f

## Overflow e Underflow

Na precisão dupla do IEEE-754, o maior valor finito é aproximadamente

$$
x_{\max}\approx1{,}8\times10^{308}.
$$

A menor magnitude positiva normal é aproximadamente

$$
x_{\min,\text{normal}}\approx2{,}2\times10^{-308}.
$$

Abaixo dessa região aparecem os números **subnormais**, permitindo representação gradual até aproximadamente

$$
4{,}9\times10^{-324}.
$$

- (a) Overflow

Para

$$
f(n)=10^n,
$$

temos:

$$
10^{307},\quad10^{308},\quad10^{309},\quad10^{310}.
$$

Como

$$
10^{308}<x_{\max}
$$

e

$$
10^{309}>x_{\max},
$$

o overflow ocorre a partir de

$$
\boxed{n=309}.
$$

- (b) Underflow

Para

$$
g(n)=10^{-n},
$$

temos:

$$
10^{-307},\quad10^{-308},\quad10^{-309},\quad10^{-310}.
$$

Os valores $10^{-307}$ e $10^{-308}$ estão próximos da transição
entre números normais e subnormais.

Os valores $10^{-309}$ e $10^{-310}$ são **subnormais**, mas ainda
diferentes de zero.

Portanto, nesses exemplos ocorre perda da faixa normal, mas não
necessariamente representação como zero.

- (e) Conceitos

**Erro de arredondamento:** diferença causada pela representação aproximada
de um número ou operação.

$$
E=|x-\hat{x}|.
$$

**Overflow:** resultado com magnitude maior que o maior valor representável.

**Underflow:** resultado com magnitude muito pequena, entrando na região
subnormal ou, em valores ainda menores, sendo representado como zero.

In [18]:
import math

print("Python\n")

print("Overflow:")
for n in [307, 308, 309, 310]:
    try:
        x = 10.0 ** n
        print(f"10^{n} = {x}")
    except OverflowError:
        print(f"10^{n} = OverflowError")

print("\nUnderflow:")
for n in [307, 308, 309, 310]:
    x = 10.0 ** (-n)
    print(f"10^-{n} = {x:.17e}")

Python

Overflow:
10^307 = 1e+307
10^308 = 1e+308
10^309 = OverflowError
10^310 = OverflowError

Underflow:
10^-307 = 9.99999999999999909e-308
10^-308 = 9.99999999999999909e-309
10^-309 = 1.00000000000000189e-309
10^-310 = 9.99999999999996945e-311
